# HAM10000 · Baby Dragon Hatchling · Hebbian explainability bundle

Trains a skin lesion classifier on HAM10000 (ISIC 2018, 7 classes) using hatchvision,
records Hebbian co-activation, clusters neurons into concepts, grounds concepts in
epidemiological metadata (sex, age group, localization - 21 binary attributes),
and exports the in-browser inference bundle.

Research use only - not for clinical diagnosis.

How to run on Kaggle:
1. Add Input: kmader/skin-lesion-analysis-toward-melanoma-detection
2. Settings: Accelerator GPU, Internet On
3. Run All (~25-40 min on T4)
4. Download /kaggle/working/bundle.zip, unzip into webapp/, redeploy

In [ ]:
!git clone --depth 1 -b main https://github.com/LarsGroep/DragonHatchling.git
%cd DragonHatchling
!pip install -q onnx onnxruntime onnxconverter-common
import sys, torch
sys.path.insert(0, ".")
import hatchvision
print("hatchvision", hatchvision.__version__, "torch", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
DATASET = "ham10000"
BACKBONE = "hybrid"
EPOCHS = 30
BATCH_SIZE = 32
LR = 3e-4
LR_CYCLE = 10
MAX_UNITS = 512
N_CONCEPTS = 16
PROBE_IMAGES = 512
VAL_RATIO = 0.15
SEED = 42
BACKBONE_KWARGS = dict(encoder="resnet50", pretrained=True, freeze_encoder=True, neuron_dim=512)
IMAGE_SIZE = 224

In [ ]:
from pathlib import Path
def find_ham_root():
    for p in Path("/kaggle/input").rglob("HAM10000_metadata.csv"):
        return p.parent
    return None
root = find_ham_root()
if root is None:
    raise RuntimeError("HAM10000_metadata.csv not found. Attach: kmader/skin-lesion-analysis-toward-melanoma-detection")
print("HAM10000 root:", root)
for d in sorted(d for d in root.iterdir() if d.is_dir()):
    n = sum(1 for _ in d.glob("*.jpg"))
    if n: print(f"  {d.name}: {n} images")

In [ ]:
from hatchvision import HebbianFeatureMemory, TrainConfig, Trainer, build_loader, create_model
from hatchvision.engine import compute_class_weights
data = build_loader(DATASET, root=str(root), image_size=IMAGE_SIZE, val_ratio=VAL_RATIO, seed=SEED)
train_loader, val_loader = data.dataloaders(batch_size=BATCH_SIZE, num_workers=2)
print(data.spec.num_classes, "classes:", ", ".join(data.spec.class_names))
print(f"train {len(train_loader.dataset)} val {len(val_loader.dataset)}")
print(f"attributes: {len(data.attribute_names() or [])}")

In [ ]:
train_ds = train_loader.dataset
all_labels = [int(train_ds[i][1]) for i in range(len(train_ds))]
class_weights = compute_class_weights(all_labels, data.spec.num_classes)
for name, w in zip(data.spec.class_names, class_weights):
    print(f"  {name}: weight {w:.2f}")

In [ ]:
model = create_model(BACKBONE, data.spec, **BACKBONE_KWARGS)
memory = HebbianFeatureMemory(model, num_classes=data.spec.num_classes, max_units=MAX_UNITS)
print("Hebbian memory:", list(model.hebbian_layers()))
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {n/1e6:.1f} M (encoder frozen)")

In [ ]:
trainer = Trainer(
    model,
    TrainConfig(epochs=EPOCHS, lr=LR, log_every=50, class_weights=class_weights, lr_cycle_epochs=LR_CYCLE),
    memory,
)
history = trainer.fit(train_loader, val_loader)
best_val = max(history["val_acc"])
print(f"best val accuracy: {best_val:.3f} (epoch {history['val_acc'].index(best_val)+1})")

## Concept clustering and epidemiological grounding

Units that fired together are clustered into Hebbian concepts.
Each concept is correlated with 21 epidemiological attributes
(sex, age group, localization) over a probe set to get meaningful labels.

In [ ]:
from hatchvision.explain import cluster_concepts, find_exemplars, ground_concepts
layer = memory.layer_names[-1]
concepts = cluster_concepts(memory, layer, data.spec.class_names, n_concepts=N_CONCEPTS)
probe = data.probe_batch(PROBE_IMAGES)
find_exemplars(concepts, memory, model, probe)
attr_names = data.attribute_names()
attr_matrix = data.probe_attributes(probe.shape[0])
if attr_names and attr_matrix is not None:
    ground_concepts(concepts, memory, model, probe, attr_matrix, attr_names)
    print(f"grounded {sum(1 for c in concepts if c.attributes)}/{len(concepts)} concepts")
for c in concepts:
    attrs = " / ".join(list(c.attributes)[:3]) if c.attributes else "-"
    top = max(c.class_affinity, key=c.class_affinity.get) if c.class_affinity else "-"
    print(f"[{c.concept_id:>2}] {len(c.units):>3} units  coh {c.coherence:.2f}  {attrs}  top: {top}")

In [ ]:
import matplotlib.pyplot as plt
from hatchvision.explain import denormalize
mean, std = data.spec.normalization()
show = [c for c in concepts if c.exemplars][:6]
n_ex = 5
fig, axes = plt.subplots(len(show), n_ex, figsize=(3*n_ex, 2.6*len(show)))
if len(show) == 1: axes = [axes]
for row_axes, c in zip(axes, show):
    for ax, idx in zip(row_axes, c.exemplars[:n_ex]):
        img = denormalize(probe[idx:idx+1], mean, std)[0]
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
        ax.axis("off")
    row_axes[0].set_title(c.label, loc="left", fontsize=9, fontweight="bold")
plt.suptitle("Exemplar dermoscopy images per Hebbian concept", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS + 1)
ax1.plot(ep, history["train_loss"], label="train"); ax1.plot(ep, history["val_loss"], label="val")
ax1.set_xlabel("epoch"); ax1.set_ylabel("weighted CE loss"); ax1.set_title("Loss"); ax1.legend()
ax2.plot(ep, history["train_acc"], label="train"); ax2.plot(ep, history["val_acc"], label="val")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
ax2.set_title(f"Accuracy (best val={max(history['val_acc']):.3f})"); ax2.legend()
plt.tight_layout(); plt.show()

## Export the web-app bundle

In [ ]:
from pathlib import Path
import torch
from hatchvision.export import export_ivgraph, export_onnx_bundle, export_explain_pack
OUT = Path("/kaggle/working/bundle")
OUT.mkdir(parents=True, exist_ok=True)
export_ivgraph(memory, concepts, layer, data.spec.class_names, OUT / "graph.json",
               meta={"dataset": data.spec.name, "backbone": BACKBONE, "val_acc": round(max(history["val_acc"]), 4)})
export_explain_pack(memory, layer, data.spec.class_names, OUT / "explain.json", model=model, background=probe[:64])
export_onnx_bundle(model.cpu(), memory, data.spec, OUT, fp16=True, explain_file="explain.json", extra_meta={"backbone": BACKBONE})
torch.save(memory.state_dict(), OUT / "hebbian_state.pt")
import shutil
shutil.make_archive("/kaggle/working/bundle", "zip", OUT)
import subprocess; subprocess.run(["ls", "-lh", "/kaggle/working/bundle", "/kaggle/working/bundle.zip"])
print("Done. Download bundle.zip, unzip into webapp/, run: cd webapp && npx vercel deploy --prod")

In [ ]:
try:
    import onnxruntime, numpy as np
    sess = onnxruntime.InferenceSession(str(OUT / "model.onnx"), providers=["CPUExecutionProvider"])
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    model.eval()
    with torch.no_grad():
        ref = model(x).numpy()
    out = sess.run(["logits"], {"images": x.numpy()})[0]
    diff = float(np.abs(out - ref).max())
    print(f"max |onnx - torch| = {diff:.4f} ({chr(39)}OK{chr(39) if diff < 0.5 else chr(39)}WARNING{chr(39)}")
    import json as _json
    manifest = _json.loads((OUT / "manifest.json").read_text())
    n_cls = len(manifest.get("class_names", manifest.get("num_classes", [])))
    print(f"manifest: {manifest[chr(39)]dataset{chr(39)]} {n_cls} classes image_size {manifest[chr(39)]image_size{chr(39)]}")
except Exception as e:
    print(f"ONNX sanity check skipped: {e}")
print("Training complete. bundle.zip is ready in /kaggle/working/")
